# Tutorial 04: S1-GRiTS Multi-Year January Mosaic — Greater Bay Area

**Purpose:** Produce a panel figure (2017–2026, Monthly composites) for the dataset,  

---

## Workflow

1. **VRT generation** — one `s1grits mosaic create` call per year (CLI, captured in-notebook)
2. **Global percentile derivation** — computed once from the 2026-01 VRT (reference year)
3. **3×3 panel figure** — each panel loads, clips (optional boundary), normalises with fixed stretch, and renders

---

## Dataset

| Field | Value |
|-------|-------|
| Dataset | `GBA_zwhoo_20260411_hARDCp` |
| Direction | `ASCENDING` |
| Years shown | 2017 – 2026 (January) |
| Layout | x rows × y columns |
| Stretch reference | VRT (EPSG:4326) |

## 1. Imports

In [1]:
import re
import warnings
from pathlib import Path

import numpy as np
import rasterio
import rioxarray
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from osgeo import gdal

warnings.filterwarnings("ignore")
plt.rcParams["font.sans-serif"] = ["Arial", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

print("INFO - Imports OK")

INFO - Imports OK


## 2. Configuration

In [2]:
# =============================================================================
# Paths
# =============================================================================

# -----------------------------------------------------------------------------
# (1) Mission: China, Guangd Dong, GBA
# OUTPUT_DIR    = r"D:\\QGIS\\s1grits-dataset\\GBA_zwhoo_20260411_hARDCp"
# DIRECTION     = "ASCENDING"

# MOSAIC_DIR    = r"D:\\QGIS\\S1-GRiTS-Figures\\Figure_GBA\\VRT"
# FIGURE_DIR    = r"D:\\QGIS\\S1-GRiTS-Figures\\Figure_GBA"

# # Optional boundary shapefile (EPSG:4326). Set to None to skip clipping.
# BOUNDARY_SHP = r"D:\\QGIS\\dataset\\GBA\\GBA_whole.shp"
# Study_Area = "GBA"

# BOUNDARY_SHP =  r"D:\\QGIS\\dataset\\GBA\\Guangdong_zhuhai_withoutIsland.shp"
# Study_Area = "Guangdong Zhuhai"
# -----------------------------------------------------------------------------

# -----------------------------------------------------------------------------
# （2）任务2：澳大利亚 PortHedland
OUTPUT_DIR    = r"D:\QGIS\s1grits-dataset\Astralia_PortHedland_DESCENDING_hARDCp"
DIRECTION     = "DESCENDING"

MOSAIC_DIR    = r"D:\QGIS\S1-GRiTS-Figures\Figure_PortHedland_AUS\Mosaic"
FIGURE_DIR    = r"D:\\QGIS\\S1-GRiTS-Figures\\Figure_PortHedland_AUS"

# Optional boundary shapefile (EPSG:4326). Set to None to skip clipping.
BOUNDARY_SHP = r"D:\\QGIS\\dataset\\shapefile\\australia_adm2_PortHedland.shp"

Study_Area = "PortHedland"

# -----------------------------------------------------------------------------
# (3) 任务3: 德国 Bayern
# OUTPUT_DIR    = r"D:\QGIS\s1grits-dataset\Gemany_Bayern_DESCENDING_hARDCp"
# DIRECTION     = "DESCENDING"

# MOSAIC_DIR    = r"D:\QGIS\S1-GRiTS-Figures\Figure_Beyern_DEU\Mosaic"
# FIGURE_DIR    = r"D:\QGIS\S1-GRiTS-Figures\Figure_Beyern_DEU"

# # Optional boundary shapefile (EPSG:4326). Set to None to skip clipping.
# BOUNDARY_SHP = r"D:\\QGIS\\dataset\\shapefile\\germany_adm1_downloaded_nurnberg.shp"

# Study_Area = "Bayern"
# -----------------------------------------------------------------------------

# =============================================================================
# Month configuration
# =============================================================================
# Month abbreviation lookup: zero-padded number string -> English abbreviation
MONTH_ABBR: dict[str, str] = {
    "01": "Jan", "02": "Feb", "03": "Mar", "04": "Apr",
    "05": "May", "06": "Jun", "07": "Jul", "08": "Aug",
    "09": "Sep", "10": "Oct", "11": "Nov", "12": "Dec",
}

# User input: enter the target month as a zero-padded number (e.g. '01' for January).
# The corresponding English abbreviation (MONTH_LABEL) is derived automatically
# and used in all figure labels, filenames, and panel annotations.
_month_input = input("Enter month (01-12, default 01): ").strip() or "01"
if _month_input not in MONTH_ABBR:
    raise ValueError(f"Invalid month input '{_month_input}'. Must be '01'–'12'.")
MONTH       = _month_input              # zero-padded string, e.g. '01'
MONTH_LABEL = MONTH_ABBR[MONTH]         # English abbreviation, e.g. 'Jan'

# =============================================================================
# Processing parameters
# =============================================================================
TARGET_CRS    = "EPSG:4326"

# Years to display — 9 panels in a 3x3 grid
# YEARS         = list(range(2017, 2027))   # [2018, 2019, ..., 2026]
YEARS = [2017, 2021, 2026]

# Reference year for global stretch (percentiles derived from this VRT)
REF_YEAR      = 2026

# =============================================================================
# RGB band assignment
# Band order in 12-band COG:
#   0=VV_dB  1=VH_dB  2=Ratio  3=RVI
#   4=VV_glcm_CONTR  5=VV_glcm_IDM  6=VV_glcm_ENT  7=VV_glcm_CORR
#   8=VH_glcm_CONTR  9=VH_glcm_IDM 10=VH_glcm_ENT 11=VH_glcm_CORR
# =============================================================================
RGB_BANDS     = (0, 1, 4)                # R=VV_dB, G=VH_dB, B=VV_glcm_CONTR
BAND_NAMES    = ("VV_dB", "VH_dB", "VV_glcm_CONTR")

# =============================================================================
# Stretch & display
# =============================================================================
P_MIN         = 2
P_MAX         = 98
GAMMA         = 1.5
DOWNSAMPLE    = 3     # spatial downsampling factor for VRT loading

# Figure layout
FIGSIZE       = (24, 16)
DPI           = 100
PANEL_FONT    = 18

from pathlib import Path
Path(MOSAIC_DIR).mkdir(parents=True, exist_ok=True)
Path(FIGURE_DIR).mkdir(parents=True, exist_ok=True)

print(f"INFO - OUTPUT_DIR  : {OUTPUT_DIR}")
print(f"INFO - MOSAIC_DIR  : {MOSAIC_DIR}")
print(f"INFO - FIGURE_DIR  : {FIGURE_DIR}")
print(f"INFO - Month       : {MONTH} ({MONTH_LABEL})")
print(f"INFO - Years       : {YEARS}")
print(f"INFO - RGB bands   : {RGB_BANDS} -> {BAND_NAMES}")
print(f"INFO - Stretch     : p=[{P_MIN},{P_MAX}]  gamma={GAMMA}")

Enter month (01-12, default 01):  03


INFO - OUTPUT_DIR  : D:\QGIS\s1grits-dataset\Astralia_PortHedland_DESCENDING_hARDCp
INFO - MOSAIC_DIR  : D:\QGIS\S1-GRiTS-Figures\Figure_PortHedland_AUS\Mosaic
INFO - FIGURE_DIR  : D:\\QGIS\\S1-GRiTS-Figures\\Figure_PortHedland_AUS
INFO - Month       : 03 (Mar)
INFO - Years       : [2017, 2021, 2026]
INFO - RGB bands   : (0, 1, 4) -> ('VV_dB', 'VH_dB', 'VV_glcm_CONTR')
INFO - Stretch     : p=[2,98]  gamma=1.5


## 3. Generate VRTs via CLI

Runs `s1grits mosaic create` for every year and captures the VRT path automatically.  
Already-existing VRTs are **skipped** to avoid redundant reprojection work.

In [3]:
import subprocess
import shutil
import time
from pathlib import Path

_s1grits = shutil.which("s1grits") or "s1grits"
VRT_PATHS: dict[int, str] = {}

for year in YEARS:
    month_str = f"{year}-{MONTH}"

    # 1. Check for an existing VRT first (fast path to save time)
    existing = sorted(Path(MOSAIC_DIR).glob(f"*-{year}-{MONTH}-{DIRECTION}-*.vrt"))
    if existing:
        VRT_PATHS[year] = str(existing[0])
        print(f"INFO  [{month_str}] Found existing VRT: {existing[0].name}")
        continue

    # 2. Run the CLI tool via subprocess to generate the VRT
    print(f"INFO  [{month_str}] Running CLI...")
    _cmd = [
        _s1grits, "mosaic",
        "--month", month_str,
        "--direction", DIRECTION,
        "--output-dir", str(OUTPUT_DIR),
        "--output", str(MOSAIC_DIR),
        "--crs", str(TARGET_CRS),
    ]
    
    # Run the command. check=True ensures Python raises an error if the command fails (non-zero exit code).
    subprocess.run(_cmd, check=True) 

    # Give the file system a brief moment to finish writing the file to disk
    time.sleep(0.5)

    # 3. 🚨 Core Logic: After execution, search directly in MOSAIC_DIR for the newly generated .vrt file.
    # We use a wildcard to match files with the specific year, month, and orbit direction.
    new_vrt = sorted(Path(MOSAIC_DIR).glob(f"*-{year}-{MONTH}-{DIRECTION}-*.vrt"))
    
    if new_vrt:
        # If found, extract the absolute path and store it in the dictionary
        vrt_path = str(new_vrt[0])
        VRT_PATHS[year] = vrt_path
        print(f"INFO  [{month_str}] VRT captured successfully: {Path(vrt_path).name}")
    else:
        # If the command succeeded but the file isn't there, something is fundamentally wrong with the CLI output path
        print(f"WARN  [{month_str}] CLI finished, but VRT file was not found in {MOSAIC_DIR}!")

print(f"\nINFO - Ready VRTs: {len(VRT_PATHS)}/{len(YEARS)}")
for yr, vp in sorted(VRT_PATHS.items()):
    print(f"       {yr}: {Path(vp).name}")

INFO  [2017-03] Found existing VRT: 50KNB-50KQD-2017-03-DESCENDING-4326.vrt
INFO  [2021-03] Found existing VRT: 50KNB-50KQD-2021-03-DESCENDING-4326.vrt
INFO  [2026-03] Found existing VRT: 50KNB-50KQD-2026-03-DESCENDING-4326.vrt

INFO - Ready VRTs: 3/3
       2017: 50KNB-50KQD-2017-03-DESCENDING-4326.vrt
       2021: 50KNB-50KQD-2021-03-DESCENDING-4326.vrt
       2026: 50KNB-50KQD-2026-03-DESCENDING-4326.vrt


## 4. Derive Global Stretch from 2026-01 Reference VRT

All panels share the same `(vmin, vmax)` per band, computed once from the reference year VRT.  
This ensures **temporal comparability** across the 3×3 panel.

In [4]:
def get_vrt_percentiles(vrt_path: str, p_min: float, p_max: float,
                        rgb_bands: tuple, scale: float = 0.1) -> dict:
    """
    Compute per-band percentile stretch from a VRT using GDAL fast-read.

    Reads a spatially downsampled (~10%) version of each band to keep
    memory usage low while maintaining statistically representative
    percentile estimates.

    Args:
        vrt_path  : Path to the reference VRT (EPSG:4326).
        p_min     : Lower percentile (e.g. 2).
        p_max     : Upper percentile (e.g. 98).
        rgb_bands : Zero-based band indices for (R, G, B).
        scale     : Fraction of full resolution to read (default 0.1 = 10%).

    Returns:
        dict with keys 'r', 'g', 'b', each a (lo, hi) float tuple.
    """
    ds = gdal.Open(str(vrt_path))
    if ds is None:
        raise FileNotFoundError(f"Cannot open VRT: {vrt_path}")

    out_x = max(1, int(ds.RasterXSize * scale))
    out_y = max(1, int(ds.RasterYSize * scale))

    stats = {}
    for key, band_idx in zip(('r', 'g', 'b'), rgb_bands):
        band = ds.GetRasterBand(band_idx + 1)   # GDAL is 1-based
        data = band.ReadAsArray(
            resample_alg=gdal.GRIORA_NearestNeighbour,
            buf_xsize=out_x, buf_ysize=out_y
        ).astype(np.float32)
        nodata = band.GetNoDataValue()
        if nodata is not None:
            data[data == nodata] = np.nan
        valid = data[np.isfinite(data)]
        if valid.size > 0:
            stats[key] = (float(np.percentile(valid, p_min)),
                          float(np.percentile(valid, p_max)))
        else:
            stats[key] = (0.0, 1.0)
    ds = None
    return stats


# Compute global stretch from the reference year VRT
REF_VRT = VRT_PATHS.get(REF_YEAR)
if REF_VRT is None:
    raise RuntimeError(f"Reference VRT for {REF_YEAR} not available — check cell above")

print(f"INFO - Computing global percentiles from: {Path(REF_VRT).name}")
print(f"       Bands: {RGB_BANDS} | p=[{P_MIN}, {P_MAX}]")

GLOBAL_STRETCH = get_vrt_percentiles(REF_VRT, P_MIN, P_MAX, RGB_BANDS)

print(f"INFO - Global stretch (from {REF_YEAR}-{MONTH} VRT):")
for key, bname in zip(('r', 'g', 'b'), BAND_NAMES):
    lo, hi = GLOBAL_STRETCH[key]
    print(f"       {key.upper()} ({bname:>12s}): [{lo:+.3f}, {hi:+.3f}]")

INFO - Computing global percentiles from: 50KNB-50KQD-2026-03-DESCENDING-4326.vrt
       Bands: (0, 1, 4) | p=[2, 98]
INFO - Global stretch (from 2026-03 VRT):
       R (       VV_dB): [-20.010, -7.198]
       G (       VH_dB): [-34.363, -14.611]
       B (VV_glcm_CONTR): [+0.000, +0.300]


## 5. Helper — Load One VRT Panel

Loads, downsamples, optionally clips, normalises, and returns an RGBA array  
ready for `ax.imshow()`.

In [5]:
def _norm(arr: np.ndarray, lo: float, hi: float, gamma: float) -> np.ndarray:
    """Clip → linear rescale → gamma correction → [0, 1]."""
    arr = arr.astype(np.float32, copy=False)
    if hi <= lo:
        return np.zeros_like(arr)
    out = np.clip((arr - lo) / (hi - lo), 0.0, 1.0)
    if gamma != 1.0:
        out = np.power(out, gamma)
    out[~np.isfinite(arr)] = np.nan
    return out


def load_panel_rgba(
    vrt_path: str,
    rgb_bands: tuple,
    stretch: dict,
    gamma: float,
    downsample: int = 3,
    boundary_gdf=None,
) -> tuple:
    """
    Load one VRT as an RGBA numpy array for panel plotting.

    Args:
        vrt_path    : Path to mosaic VRT (EPSG:4326).
        rgb_bands   : Zero-based (R, G, B) band indices.
        stretch     : Dict {'r': (lo,hi), 'g': (lo,hi), 'b': (lo,hi)}.
        gamma       : Gamma correction exponent.
        downsample  : Spatial downsampling factor (coarsen step).
        boundary_gdf: Optional GeoDataFrame for spatial clipping (EPSG:4326).

    Returns:
        (rgba, extent) where extent = [xmin, xmax, ymin, ymax] in EPSG:4326.
    """
    # Load synchronously (no Dask) to ensure each panel reads its own VRT immediately.
    # Using chunks="auto" with Dask caused all panels to render the same data
    # because lazy evaluation deferred computation until after the loop completed.
    ds = rioxarray.open_rasterio(vrt_path, masked=True)

    # Downsample before clipping for performance
    if downsample > 1:
        ds = ds.coarsen(x=downsample, y=downsample, boundary="trim").mean()

    # Optional boundary clip
    if boundary_gdf is not None:
        try:
            ds = ds.rio.clip(boundary_gdf.geometry, ds.rio.crs,
                             drop=True, all_touched=False)
        except Exception as e:
            print(f"WARN - Clipping failed ({e}), using full extent")

    xmin = float(ds.x.min())
    xmax = float(ds.x.max())
    ymin = float(ds.y.min())
    ymax = float(ds.y.max())
    extent = [xmin, xmax, ymin, ymax]

    idx_r, idx_g, idx_b = rgb_bands
    r_data = ds.isel(band=idx_r).values
    g_data = ds.isel(band=idx_g).values
    b_data = ds.isel(band=idx_b).values
    ds.close()

    valid = np.isfinite(r_data) & np.isfinite(g_data) & np.isfinite(b_data)
    alpha = valid.astype(np.float32)

    r_norm = _norm(r_data, stretch['r'][0], stretch['r'][1], gamma)
    g_norm = _norm(g_data, stretch['g'][0], stretch['g'][1], gamma)
    b_norm = _norm(b_data, stretch['b'][0], stretch['b'][1], gamma)

    rgba = np.dstack((
        np.nan_to_num(r_norm, nan=0.0),
        np.nan_to_num(g_norm, nan=0.0),
        np.nan_to_num(b_norm, nan=0.0),
        alpha,
    ))
    return rgba, extent


print("INFO - load_panel_rgba() defined")

INFO - load_panel_rgba() defined


# Phase 1 — Generate per-year raw RGBA panel PNGs

In [6]:
# =============================================================================
# Phase 1 — Generate per-year raw RGBA panel PNGs (no gridlines)
#
# Each PNG stores only the SAR false-color image pixels (+ alpha).
# Cartopy gridlines, coastlines and labels are added in Phase 2 so that
# the 3x3 composite has a single, consistent coordinate system and
# gridline labels appear only on the outer edges of the grid.
#
# A JSON sidecar file is saved alongside each PNG storing the geographic
# extent [xmin, xmax, ymin, ymax] needed by Phase 2 to geo-register the image.
#
# Uses Dask chunked I/O for large VRTs. Calling .compute() immediately after
# each band selection forces evaluation for that specific VRT before closing
# the dataset — avoiding the deferred-evaluation bug from the original loop.
# =============================================================================
import json as _json
import numpy as np
import geopandas as gpd
import matplotlib
matplotlib.use("Agg")   # non-interactive backend
import matplotlib.pyplot as plt
import rioxarray
from PIL import Image
from pathlib import Path

PANEL_PNG_DIR = Path(FIGURE_DIR) / "panel_cache"
PANEL_PNG_DIR.mkdir(parents=True, exist_ok=True)

# Sanitize Study_Area for use in filenames (replace spaces with underscores)
study_tag = Study_Area.replace(" ", "_")

YEARS_PLOT = [y for y in YEARS[::-1] if y >= 2017]   # [2026, 2025, ..., 2018] — skip 2016/2017 (no VRT data)

# Load boundary for clipping
boundary_gdf = None
if BOUNDARY_SHP is not None:
    boundary_gdf = gpd.read_file(BOUNDARY_SHP)
    if str(boundary_gdf.crs) != "EPSG:4326":
        boundary_gdf = boundary_gdf.to_crs("EPSG:4326")
    print(f"INFO - Boundary loaded: {Path(BOUNDARY_SHP).name}")


def save_raw_panel_png(year: int, vrt_path: str,
                       out_png: Path, out_json: Path) -> None:
    """
    Load VRT with Dask, clip, normalise, and save as a lossless RGBA PNG.

    No cartographic elements (gridlines, coastlines) are drawn here;
    they are added in Phase 2 so the 3x3 composite shares a single
    coordinate system.  Geographic extent is stored in a JSON sidecar
    so Phase 2 can geo-register each panel correctly.
    """
    # Open with Dask chunks — safe because .compute() is called per-band below
    ds = rioxarray.open_rasterio(vrt_path,
                                 chunks={"x": 2048, "y": 2048},
                                 masked=True)

    if DOWNSAMPLE > 1:
        ds = ds.coarsen(x=DOWNSAMPLE, y=DOWNSAMPLE, boundary="trim").mean()

    if boundary_gdf is not None:
        try:
            ds = ds.rio.clip(boundary_gdf.geometry, ds.rio.crs,
                             drop=True, all_touched=False)
        except Exception as e:
            print(f"WARN - Clip failed for {year}: {e}")

    xmin = float(ds.x.min())
    xmax = float(ds.x.max())
    ymin = float(ds.y.min())
    ymax = float(ds.y.max())
    extent = [xmin, xmax, ymin, ymax]

    idx_r, idx_g, idx_b = RGB_BANDS
    # .compute() forces Dask evaluation NOW for this specific VRT
    r_data = ds.isel(band=idx_r).compute().values.astype(np.float32)
    g_data = ds.isel(band=idx_g).compute().values.astype(np.float32)
    b_data = ds.isel(band=idx_b).compute().values.astype(np.float32)
    ds.close()

    valid = np.isfinite(r_data) & np.isfinite(g_data) & np.isfinite(b_data)
    alpha = valid.astype(np.float32)

    r_norm = _norm(r_data, GLOBAL_STRETCH["r"][0], GLOBAL_STRETCH["r"][1], GAMMA)
    g_norm = _norm(g_data, GLOBAL_STRETCH["g"][0], GLOBAL_STRETCH["g"][1], GAMMA)
    b_norm = _norm(b_data, GLOBAL_STRETCH["b"][0], GLOBAL_STRETCH["b"][1], GAMMA)

    rgba = np.dstack((
        np.nan_to_num(r_norm, nan=0.0),
        np.nan_to_num(g_norm, nan=0.0),
        np.nan_to_num(b_norm, nan=0.0),
        alpha,
    ))

    # Save lossless RGBA PNG via PIL
    rgba_u8 = (rgba * 255).clip(0, 255).astype(np.uint8)
    Image.fromarray(rgba_u8, mode="RGBA").save(str(out_png))

    # Save geographic extent as JSON sidecar
    with open(out_json, "w") as _f:
        _json.dump({"extent": extent, "year": year, "month": MONTH}, _f)


# ---------------------------------------------------------------------------
# Run Phase 1 for all years
# ---------------------------------------------------------------------------
PANEL_PNG_PATHS: dict[int, Path] = {}

for year in YEARS_PLOT:
    out_png  = PANEL_PNG_DIR / f"panel_{year}_{MONTH}_{DIRECTION}_{study_tag}.png"
    out_json = PANEL_PNG_DIR / f"panel_{year}_{MONTH}_{DIRECTION}_{study_tag}.json"
    PANEL_PNG_PATHS[year] = out_png

    if out_png.exists() and out_json.exists():
        print(f"INFO  [{year}-{MONTH}] Cache hit — skipping: {out_png.name}")
        continue

    vrt_path = VRT_PATHS.get(year)
    if vrt_path is None:
        print(f"WARN  [{year}-{MONTH}] No VRT found — skipping")
        continue

    print(f"INFO  [{year}-{MONTH}] Generating raw panel PNG (Dask) ...")
    save_raw_panel_png(year, vrt_path, out_png, out_json)
    print(f"INFO  [{year}-{MONTH}] Saved: {out_png.name}")

print(f"\nINFO - Phase 1 complete. {len(PANEL_PNG_PATHS)} panel PNGs ready in {PANEL_PNG_DIR}")
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")   # restore inline backend


INFO - Boundary loaded: australia_adm2_PortHedland.shp
INFO  [2026-03] Generating raw panel PNG (Dask) ...
INFO  [2026-03] Saved: panel_2026_03_DESCENDING_PortHedland.png
INFO  [2021-03] Generating raw panel PNG (Dask) ...
INFO  [2021-03] Saved: panel_2021_03_DESCENDING_PortHedland.png
INFO  [2017-03] Generating raw panel PNG (Dask) ...
INFO  [2017-03] Saved: panel_2017_03_DESCENDING_PortHedland.png

INFO - Phase 1 complete. 3 panel PNGs ready in D:\QGIS\S1-GRiTS-Figures\Figure_PortHedland_AUS\panel_cache


## 6. Build 3×3 Panel Figure

Panels are arranged **chronologically left-to-right, top-to-bottom**:

| | Col 1 | Col 2 | Col 3 |
|---|---|---|---|
| **Row 1** | 2018-01 | 2019-01 | 2020-01 |
| **Row 2** | 2021-01 | 2022-01 | 2023-01 |
| **Row 3** | 2024-01 | 2025-01 | 2026-01 |

## 6. Two-Phase Fast Rendering

Reading the full VRT mosaic (large geographic extent) nine times inside a single
loop is slow. The two-phase strategy below decouples heavy I/O from figure
assembly:

- **Phase 1** (`cell-13a`): For each year, open the VRT with Dask chunked I/O,
  downsample, clip, normalise, and save a single RGBA PNG to `PANEL_PNG_DIR`.
  Already-existing PNGs are skipped, so re-runs are instant.
- **Phase 2** (`cell-13b`): Read the small cached PNGs with `matplotlib.image.imread`
  and assemble the composite. No VRT I/O; layout tweaks are near-instant.

> **Why Dask in Phase 1 is safe:** each panel PNG is computed and written to disk
> *before* moving to the next year, so there is no deferred-evaluation bug.

In [7]:
# =============================================================================
# Phase 2 — Assemble 3x3 figure from cached raw RGBA PNGs
#
# Final Optimizations: 
# - DRAFT_MODE enabled for ~3 second layout testing.
# - layout="compressed" for absolute gapless panel stitching.
# - Shared global extent to prevent axis label misalignment.
# - Added '\n' to suptitle to prevent clipping of the top-row Y-axis label.
# =============================================================================
import json as _json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from pathlib import Path
from PIL import Image
import geopandas as gpd
import time

# =========================================================
# ⚙️ Core Layout Switch
# =========================================================
DRAFT_MODE = True  # Set to True for fast layout testing; False for final high-res output

PANEL_PNG_DIR = Path(FIGURE_DIR) / "panel_cache"
N_ROWS, N_COLS = 3, 3
plot_crs = ccrs.PlateCarree()

print(f"\n{'='*50}")
print(f"🚀 Assembling 3x3 mosaic (Current Mode: {'[DRAFT]' if DRAFT_MODE else '[FINAL HIGH-RES]'})")
print(f"{'='*50}")
t_start = time.time()

# 1. Pre-load vector boundary (if provided)
boundary_gdf = None
if BOUNDARY_SHP is not None:
    boundary_gdf = gpd.read_file(BOUNDARY_SHP).to_crs("EPSG:4326")

# 2. Initialize figure (using 'compressed' layout to eliminate column gaps)
fig, axes = plt.subplots(
    N_ROWS, N_COLS,
    figsize=FIGSIZE, 
    subplot_kw={"projection": plot_crs},
    facecolor="white",
    layout="compressed"  # <--- Core change: forces absolute gapless stitching
)
fig.patch.set_facecolor("white")

# Lock shared extent for all subplots (solves minor Y-axis label jitter)
shared_extent_pad = None

# =========================================================
# 3. Render each panel
# =========================================================
for panel_idx, year in enumerate(YEARS_PLOT[:]):
    row, col = divmod(panel_idx, N_COLS)
    ax = axes[row, col]

    png_path  = PANEL_PNG_DIR / f"panel_{year}_{MONTH}_{DIRECTION}_{study_tag}.png"
    json_path = PANEL_PNG_DIR / f"panel_{year}_{MONTH}_{DIRECTION}_{study_tag}.json"

    if not png_path.exists() or not json_path.exists():
        ax.text(0.5, 0.5, f"{year}-{MONTH}\nNo cache",
                ha="center", va="center", transform=ax.transAxes,
                fontsize=PANEL_FONT, color="red")
        ax.axis("off")
        continue

    # Read JSON for physical coordinate extent (mandatory, millisecond speed)
    with open(json_path) as _f:
        meta = _json.load(_f)
    xmin, xmax, ymin, ymax = meta["extent"]
    extent = [xmin, xmax, ymin, ymax]

    # Initialize and lock shared boundary
    if shared_extent_pad is None:
        xpad = (xmax - xmin) * 0.05
        ypad = (ymax - ymin) * 0.05
        shared_extent_pad = [xmin - xpad, xmax + xpad, ymin - ypad, ymax + ypad]
    
    # Force current ax to use the shared boundary
    ax.set_extent(shared_extent_pad, crs=plot_crs)

    # --- Core Rendering Branch ---
    if not DRAFT_MODE:
        # [High-Res Mode]: Load basemap and real image pixels (time-consuming)
        rgba = np.array(Image.open(str(png_path)).convert("RGBA"),
                        dtype=np.float32) / 255.0
        
        ax.add_feature(cfeature.OCEAN, facecolor="#cde8f5", zorder=1)
        ax.add_feature(cfeature.LAND,  facecolor="#f5f5f0", zorder=2)
        ax.imshow(rgba, extent=extent, transform=plot_crs,
                  origin="upper", interpolation="bilinear", zorder=5)
        ax.add_feature(cfeature.COASTLINE,
                       linewidth=0.8, edgecolor="#333333", alpha=0.8, zorder=10)
        
        if boundary_gdf is not None:
            ax.add_geometries(boundary_gdf.geometry, crs=plot_crs,
                              facecolor="none", edgecolor="black",
                              linewidth=0.6, zorder=15)
    else:
        # [Draft Mode]: Grey placeholder for extremely fast layout validation
        ax.set_facecolor("#eeeeee")
        ax.text(0.5, 0.5, "DRAFT", transform=ax.transAxes, 
                ha="center", va="center", fontsize=30, color="grey", alpha=0.4, zorder=5)
        
        # Draw boundary even in draft mode for spatial reference
        if boundary_gdf is not None:
            ax.add_geometries(boundary_gdf.geometry, crs=plot_crs,
                              facecolor="none", edgecolor="#aaaaaa",
                              linewidth=0.8, linestyle="--", zorder=15)

    # --- Gridlines and Layout Scales ---
    gl = ax.gridlines(
        draw_labels=True, linewidth=0.5,
        color="grey", alpha=0.5, linestyle="--", zorder=20
    )
    
    # Keep coordinate labels only on outer edges
    gl.top_labels    = False
    gl.right_labels  = False
    gl.left_labels   = (col == 0)            # Show latitude only on the leftmost column
    gl.bottom_labels = (row == N_ROWS - 1)   # Show longitude only on the bottom row
    
    gl.xlocator      = mticker.MultipleLocator(1.0)
    gl.ylocator      = mticker.MultipleLocator(1.0)
    gl.xformatter    = LONGITUDE_FORMATTER
    gl.yformatter    = LATITUDE_FORMATTER
    gl.xlabel_style  = {"size": (PANEL_FONT - 4) * 1.8}
    gl.ylabel_style  = {"size": (PANEL_FONT - 4) * 1.8,
                        "rotation": 90, "ha": "center", "va": "center"}

    # Top-right Year/Month label
    ax.text(0.98, 0.98, f"{year} {MONTH_LABEL}",
            transform=ax.transAxes, ha="right", va="top",
            fontsize=PANEL_FONT + 8, color="black",
            fontweight="normal", zorder=30)

    # Panel border
    ax.spines["geo"].set_linewidth(1.2)
    ax.spines["geo"].set_edgecolor("black")

# =========================================================
# 4. Figure Title and Output
# =========================================================
# Added \n at the end of the title to create space for the top-row Y-axis label (prevents culling)
fig.suptitle(
    f"Greater Bay Area \u2014 Sentinel-1 Monthly Composite"
    f" ({MONTH_LABEL}, {min(YEARS)}\u2013{max(YEARS)})\n"
    f"R={BAND_NAMES[0]}  G={BAND_NAMES[1]}  B={BAND_NAMES[2]}  "
    f"| Orbit: {DIRECTION} | Stretch: p=[{P_MIN},{P_MAX}], gamma={GAMMA}"
    f" (reference: {REF_YEAR}-{MONTH})\n",  # <--- Crucial newline here
    fontsize=PANEL_FONT + 6, fontweight="bold"
)

# Dynamically adjust save DPI and file suffix based on mode
save_dpi = 72 if DRAFT_MODE else DPI
suffix = "DRAFT" if DRAFT_MODE else "FINAL"
out_png = Path(FIGURE_DIR) / f"{Study_Area}_{MONTH_LABEL}_{YEARS[0]}-{YEARS[-1]}_{DIRECTION}_3x3_{suffix}.png"

print(f"\n⏳ Rendering and saving to disk (DPI: {save_dpi}) ...")
fig.savefig(out_png, dpi=save_dpi, bbox_inches="tight",
            facecolor="white", pad_inches=0.03)

t_end = time.time()
print(f"✅ Done! Total elapsed time: {t_end - t_start:.2f} seconds")
print(f"📂 File path: {out_png}")

plt.show()


🚀 Assembling 3x3 mosaic (Current Mode: [DRAFT])

⏳ Rendering and saving to disk (DPI: 72) ...
✅ Done! Total elapsed time: 0.35 seconds
📂 File path: D:\QGIS\S1-GRiTS-Figures\Figure_PortHedland_AUS\PortHedland_Mar_2017-2026_DESCENDING_3x3_DRAFT.png


<Figure size 2400x1600 with 9 Axes>

In [8]:
# =============================================================================
# Phase 2 — Assemble 3x3 figure from cached raw RGBA PNGs
#
# Final High-Resolution Mode:
# - DRAFT_MODE = False (Loads actual satellite imagery and Cartopy vectors)
# - DPI = 300 (Publication quality)
# - layout="compressed" for absolute gapless panel stitching.
# - Shared global extent to prevent axis label misalignment.
# =============================================================================
import json as _json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from pathlib import Path
from PIL import Image
import geopandas as gpd
import time

# =========================================================
# ⚙️ Core Layout Switch & Quality Settings
# =========================================================
DRAFT_MODE = False  # Set to False to perform the actual data rendering
FINAL_DPI = 600     # Force publication-quality resolution

PANEL_PNG_DIR = Path(FIGURE_DIR) / "panel_cache"
N_ROWS, N_COLS = 3, 3
plot_crs = ccrs.PlateCarree()

print(f"\n{'='*50}")
print(f"🚀 Assembling 3x3 mosaic (Current Mode: {'[DRAFT]' if DRAFT_MODE else '[FINAL HIGH-RES]'})")
print(f"{'='*50}")
t_start = time.time()

# 1. Pre-load vector boundary (if provided)
boundary_gdf = None
if BOUNDARY_SHP is not None:
    boundary_gdf = gpd.read_file(BOUNDARY_SHP).to_crs("EPSG:4326")

# 2. Initialize figure (using 'compressed' layout to eliminate column gaps)
fig, axes = plt.subplots(
    N_ROWS, N_COLS,
    figsize=FIGSIZE, 
    subplot_kw={"projection": plot_crs},
    facecolor="white",
    layout="compressed"  
)
fig.patch.set_facecolor("white")

# Lock shared extent for all subplots (solves minor Y-axis label jitter)
shared_extent_pad = None

# =========================================================
# 3. Render each panel
# =========================================================
for panel_idx, year in enumerate(YEARS_PLOT[:]):
    row, col = divmod(panel_idx, N_COLS)
    ax = axes[row, col]

    png_path  = PANEL_PNG_DIR / f"panel_{year}_{MONTH}_{DIRECTION}_{study_tag}.png"
    json_path = PANEL_PNG_DIR / f"panel_{year}_{MONTH}_{DIRECTION}_{study_tag}.json"

    if not png_path.exists() or not json_path.exists():
        ax.text(0.5, 0.5, f"{year}-{MONTH}\nNo cache",
                ha="center", va="center", transform=ax.transAxes,
                fontsize=PANEL_FONT, color="red")
        ax.axis("off")
        continue

    # Read JSON for physical coordinate extent
    with open(json_path) as _f:
        meta = _json.load(_f)
    xmin, xmax, ymin, ymax = meta["extent"]
    extent = [xmin, xmax, ymin, ymax]

    # Initialize and lock shared boundary
    if shared_extent_pad is None:
        xpad = (xmax - xmin) * 0.05
        ypad = (ymax - ymin) * 0.05
        shared_extent_pad = [xmin - xpad, xmax + xpad, ymin - ypad, ymax + ypad]
    
    # Force current ax to use the shared boundary
    ax.set_extent(shared_extent_pad, crs=plot_crs)

    # --- Core Rendering Branch ---
    if not DRAFT_MODE:
        # [High-Res Mode]: Load basemap and real image pixels
        rgba = np.array(Image.open(str(png_path)).convert("RGBA"),
                        dtype=np.float32) / 255.0
        
        ax.add_feature(cfeature.OCEAN, facecolor="#cde8f5", zorder=1)
        ax.add_feature(cfeature.LAND,  facecolor="#f5f5f0", zorder=2)
        
        # Plot the actual SAR satellite composite
        ax.imshow(rgba, extent=extent, transform=plot_crs,
                  origin="upper", interpolation="bilinear", zorder=5)
                  
        ax.add_feature(cfeature.COASTLINE,
                       linewidth=0.8, edgecolor="#333333", alpha=0.8, zorder=10)
        
        if boundary_gdf is not None:
            ax.add_geometries(boundary_gdf.geometry, crs=plot_crs,
                              facecolor="none", edgecolor="black",
                              linewidth=0.6, zorder=15)
    else:
        # [Draft Mode]: Grey placeholder 
        ax.set_facecolor("#eeeeee")
        ax.text(0.5, 0.5, "DRAFT", transform=ax.transAxes, 
                ha="center", va="center", fontsize=30, color="grey", alpha=0.4, zorder=5)
        
        if boundary_gdf is not None:
            ax.add_geometries(boundary_gdf.geometry, crs=plot_crs,
                              facecolor="none", edgecolor="#aaaaaa",
                              linewidth=0.8, linestyle="--", zorder=15)

    # --- Gridlines and Layout Scales ---
    gl = ax.gridlines(
        draw_labels=True, linewidth=0.5,
        color="grey", alpha=0.5, linestyle="--", zorder=20
    )
    
    # Keep coordinate labels only on outer edges
    gl.top_labels    = False
    gl.right_labels  = False
    gl.left_labels   = (col == 0)            # Show latitude only on the leftmost column
    gl.bottom_labels = (row == N_ROWS - 1)   # Show longitude only on the bottom row
    
    gl.xlocator      = mticker.MultipleLocator(1.0)
    gl.ylocator      = mticker.MultipleLocator(1.0)
    gl.xformatter    = LONGITUDE_FORMATTER
    gl.yformatter    = LATITUDE_FORMATTER
    gl.xlabel_style  = {"size": (PANEL_FONT - 4) * 1.8}
    gl.ylabel_style  = {"size": (PANEL_FONT - 4) * 1.8,
                        "rotation": 90, "ha": "center", "va": "center"}

    # Top-right Year/Month label
    ax.text(0.98, 0.98, f"{year} {MONTH_LABEL}",
            transform=ax.transAxes, ha="right", va="top",
            fontsize=PANEL_FONT + 8, color="black",
            fontweight="normal", zorder=30)

    # Panel border
    ax.spines["geo"].set_linewidth(1.2)
    ax.spines["geo"].set_edgecolor("black")

# =========================================================
# 4. Figure Title and Output
# =========================================================
fig.suptitle(
    f"Greater Bay Area \u2014 Sentinel-1 Monthly Composite"
    f" ({MONTH_LABEL}, {min(YEARS)}\u2013{max(YEARS)})\n"
    f"R={BAND_NAMES[0]}  G={BAND_NAMES[1]}  B={BAND_NAMES[2]}  "
    f"| Orbit: {DIRECTION} | Stretch: p=[{P_MIN},{P_MAX}], gamma={GAMMA}"
    f" (reference: {REF_YEAR}-{MONTH})\n", 
    fontsize=PANEL_FONT + 6, fontweight="bold"
)

# Output generation
suffix = "DRAFT" if DRAFT_MODE else "FINAL"
out_png = Path(FIGURE_DIR) / f"{Study_Area}_{MONTH_LABEL}_{YEARS[0]}-{YEARS[-1]}_{DIRECTION}_3x3_{suffix}.png"

print(f"\n⏳ Rendering and saving to disk (DPI: {FINAL_DPI}). This may take a few minutes...")
fig.savefig(out_png, dpi=FINAL_DPI, bbox_inches="tight",
            facecolor="white", pad_inches=0.03)

t_end = time.time()
print(f"✅ Done! Total elapsed time: {t_end - t_start:.2f} seconds")
print(f"📂 File path: {out_png}")

plt.show()


🚀 Assembling 3x3 mosaic (Current Mode: [FINAL HIGH-RES])

⏳ Rendering and saving to disk (DPI: 600). This may take a few minutes...
✅ Done! Total elapsed time: 10.27 seconds
📂 File path: D:\QGIS\S1-GRiTS-Figures\Figure_PortHedland_AUS\GBA_Mar_2017-2026_DESCENDING_3x3_FINAL.png


<Figure size 2400x1600 with 9 Axes>

## 7. Asymmetric 4×3 Visualization Layout (One Large + Two Small Panels)

This figure is designed to highlight three selected temporal snapshots using an asymmetric multi-panel layout. Instead of a conventional 3×3 grid, we adopt a 2×2 layout where the left column contains a single large panel spanning two rows, and the right column contains two smaller panels stacked vertically. This arrangement emphasizes the primary panel while still allowing compact comparison with two additional reference years.

The three panels correspond to the first three elements in `YEARS_PLOT`:

- **Left large panel**: `YEARS_PLOT[0]`
- **Top-right panel**: `YEARS_PLOT[1]`
- **Bottom-right panel**: `YEARS_PLOT[2]`

This layout supports direct visual comparison while preserving spatial detail in the main panel.

### Conceptual Layout
```
+-------------------+-------------+
|                   |   small 1   |
|     big panel     +-------------+
|                   |   small 2   |
+-------------------+-------------+
```

In [9]:
# # =============================================================================
# # Asymmetrical 4x3 Grid Layout (All 9 Years)
# #
# # Layout Structure (Revised Order):
# # - ax_a (Top-Left)    : 2026
# # - ax_b (Mid-Left)    : 2025
# # - ax_c (Big-Right)   : 2024 (Spans rows 0-1, cols 1-2)
# # - ax_d to ax_i       : 2023 to 2018 (Rows 2-3)
# #
# # Optimizations:
# # - Y-axis Padding: gl.ypadding = 15 added to push right-axis labels outwards.
# # - Title Cleaned: Removed the reference year text.
# # - Alignment Fix: The big panel (c) receives a slightly expanded Y-extent 
# #   to perfectly align its physical top/bottom borders with the stacked small panels.
# # =============================================================================
# import geopandas as gpd
# import matplotlib.pyplot as plt
# import matplotlib.ticker as mticker
# from matplotlib.gridspec import GridSpec
# import cartopy.crs as ccrs
# import cartopy.feature as cfeature
# from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
# from pathlib import Path
# import time
# import rioxarray

# # =========================================================
# # ⚙️ Core Layout Switch & Quality Settings
# # =========================================================
# DRAFT_MODE = True  # Set to True for fast layout testing; False for final output
# FINAL_DPI = 300

# print(f"\n{'='*50}")
# print(f"🚀 Assembling 4x3 Master Grid (Mode: {'[DRAFT]' if DRAFT_MODE else '[FINAL HIGH-RES]'})")
# print(f"{'='*50}")
# t_start = time.time()

# # =========================================================
# # Load boundary once
# # =========================================================
# boundary_gdf = None
# if BOUNDARY_SHP is not None:
#     boundary_gdf = gpd.read_file(BOUNDARY_SHP)
#     if str(boundary_gdf.crs) != "EPSG:4326":
#         boundary_gdf = boundary_gdf.to_crs("EPSG:4326")
#     print(f"INFO - Boundary loaded: {Path(BOUNDARY_SHP).name}  CRS={boundary_gdf.crs}")

# YEARS_PLOT = YEARS[::-1]  
# plot_crs = ccrs.PlateCarree()

# # =========================================================
# # Figure layout: 4 Rows × 3 Cols 
# # =========================================================
# FIGSIZE4x3 = (15, 18) 
# fig = plt.figure(figsize=FIGSIZE4x3, facecolor="white", layout="compressed")

# # Set wspace/hspace to 0.0 to help the layout engine minimize gaps
# gs = GridSpec(4, 3, figure=fig, wspace=0.0, hspace=0.0)

# # REVISED ORDER: a, b (Left small), c (Right big)
# ax_a = fig.add_subplot(gs[0, 0], projection=plot_crs)      # 2026: Top-Left
# ax_b = fig.add_subplot(gs[1, 0], projection=plot_crs)      # 2025: Mid-Left
# ax_c = fig.add_subplot(gs[0:2, 1:3], projection=plot_crs)  # 2024: Big-Right (spans 2x2)

# ax_d = fig.add_subplot(gs[2, 0], projection=plot_crs)      # 2023: Row 2, Left
# ax_e = fig.add_subplot(gs[2, 1], projection=plot_crs)      # 2022: Row 2, Mid
# ax_f = fig.add_subplot(gs[2, 2], projection=plot_crs)      # 2021: Row 2, Right
# ax_g = fig.add_subplot(gs[3, 0], projection=plot_crs)      # 2020: Row 3, Left
# ax_h = fig.add_subplot(gs[3, 1], projection=plot_crs)      # 2019: Row 3, Mid
# ax_i = fig.add_subplot(gs[3, 2], projection=plot_crs)      # 2018: Row 3, Right

# axes_list = [ax_a, ax_b, ax_c, ax_d, ax_e, ax_f, ax_g, ax_h, ax_i]
# shared_extent_pad = None

# # =========================================================
# # Render each panel
# # =========================================================
# for panel_idx, (ax, year) in enumerate(zip(axes_list, YEARS_PLOT)):
#     month_str = f"{year}-{MONTH}"
#     vrt_path = VRT_PATHS.get(year)
#     panel_letter = chr(97 + panel_idx)  

#     if vrt_path is None:
#         ax.text(0.5, 0.5, f"{month_str}\nNo VRT", ha="center", va="center", 
#                 transform=ax.transAxes, fontsize=PANEL_FONT, color="red")
#         ax.axis("off")
#         continue

#     print(f"INFO  [{month_str}] Loading panel {panel_letter} (idx:{panel_idx}) ...")

#     extent = None
#     try:
#         if DRAFT_MODE:
#             if boundary_gdf is not None:
#                 minx, miny, maxx, maxy = boundary_gdf.total_bounds
#                 extent = [minx, maxx, miny, maxy]
#             else:
#                 with rioxarray.open_rasterio(vrt_path) as ds:
#                     extent = [float(ds.x.min()), float(ds.x.max()), float(ds.y.min()), float(ds.y.max())]
#         else:
#             rgba, extent = load_panel_rgba(
#                 vrt_path=vrt_path,
#                 rgb_bands=RGB_BANDS,
#                 stretch=GLOBAL_STRETCH,
#                 gamma=GAMMA,
#                 downsample=DOWNSAMPLE,
#                 boundary_gdf=boundary_gdf,
#             )
#     except Exception as e:
#         print(f"ERROR [{month_str}] {e}")
#         ax.text(0.5, 0.5, f"{month_str}\nLoad Error", ha="center", va="center", 
#                 transform=ax.transAxes, fontsize=PANEL_FONT, color="red")
#         ax.axis("off")
#         continue

#     # --- Lock global shared extent ---
#     if shared_extent_pad is None:
#         xmin, xmax, ymin, ymax = extent
#         xpad = (xmax - xmin) * 0.03
#         ypad = (ymax - ymin) * 0.03
#         shared_extent_pad = [xmin - xpad, xmax + xpad, ymin - ypad, ymax + ypad]
    
#     # 🎯 ALIGNMENT FIX: Slightly expand Y-extent for the big panel (c) 
#     # to compensate for the missing row-gap, ensuring flush top/bottom borders.
#     if panel_idx == 2:
#         y_diff = shared_extent_pad[3] - shared_extent_pad[2]
#         expansion_factor = 0.012  # Expand by ~1.2% vertically
#         big_extent_pad = [
#             shared_extent_pad[0], 
#             shared_extent_pad[1], 
#             shared_extent_pad[2] - (y_diff * expansion_factor), 
#             shared_extent_pad[3] + (y_diff * expansion_factor)
#         ]
#         ax.set_extent(big_extent_pad, crs=plot_crs)
#     else:
#         ax.set_extent(shared_extent_pad, crs=plot_crs)

#     # --- Core Rendering ---
#     if not DRAFT_MODE:
#         ax.add_feature(cfeature.OCEAN, facecolor="#cde8f5", zorder=1)
#         ax.add_feature(cfeature.LAND, facecolor="#f5f5f0", zorder=2)
#         ax.imshow(rgba, extent=extent, transform=plot_crs,
#                   origin="upper", interpolation="bilinear", zorder=5)
#         ax.add_feature(cfeature.COASTLINE, linewidth=0.8, edgecolor="#333333", alpha=0.8, zorder=10)
        
#         if boundary_gdf is not None:
#             ax.add_geometries(boundary_gdf.geometry, crs=plot_crs, facecolor="none",
#                               edgecolor="black", linewidth=0.6, zorder=15)
#     else:
#         ax.set_facecolor("#eeeeee")
#         ax.text(0.5, 0.5, "DRAFT", transform=ax.transAxes, ha="center", va="center", 
#                 fontsize=40, color="grey", alpha=0.4, zorder=5)
#         if boundary_gdf is not None:
#             ax.add_geometries(boundary_gdf.geometry, crs=plot_crs, facecolor="none", 
#                               edgecolor="#aaaaaa", linewidth=0.8, linestyle="--", zorder=15)

#     # --- Gridlines & Axis Labels ---
#     gl = ax.gridlines(draw_labels=True, linewidth=0.5, color="grey", alpha=0.5, linestyle="--", zorder=20)
    
#     # 🎯 PADDING FIX: Push Y-axis and X-axis labels away from the frame
#     gl.ypadding = 15  
#     gl.xpadding = 10  

#     gl.top_labels = False
#     gl.bottom_labels = False
#     gl.left_labels = False
#     gl.right_labels = False

#     # REVISED LABEL LOGIC based on new a,b,c ordering
#     if panel_idx == 0:     # a (2026, Top-Left)
#         gl.top_labels = True
#         gl.left_labels = True
#     elif panel_idx == 1:   # b (2025, Mid-Left)
#         gl.left_labels = True
#     elif panel_idx == 2:   # c (2024, Big-Right)
#         gl.top_labels = True
#         gl.right_labels = True
#     elif panel_idx == 3:   # d (2023, Row 2 Left)
#         gl.left_labels = True
#     elif panel_idx == 4:   # e (2022, Row 2 Mid)
#         pass # Inner panel
#     elif panel_idx == 5:   # f (2021, Row 2 Right)
#         gl.right_labels = True
#     elif panel_idx == 6:   # g (2020, Bottom Left)
#         gl.bottom_labels = True
#         gl.left_labels = True
#     elif panel_idx == 7:   # h (2019, Bottom Mid)
#         gl.bottom_labels = True
#     elif panel_idx == 8:   # i (2018, Bottom Right)
#         gl.bottom_labels = True
#         gl.right_labels = True

#     gl.xlocator = mticker.MultipleLocator(1.0)
#     gl.ylocator = mticker.MultipleLocator(1.0)
#     gl.xformatter = LONGITUDE_FORMATTER
#     gl.yformatter = LATITUDE_FORMATTER

#     # Font scaling perfectly matches the original 3x3
#     gl.xlabel_style = {"size": (PANEL_FONT - 4) * 1.8}
#     gl.ylabel_style = {"size": (PANEL_FONT - 4) * 1.8, "rotation": 90, "ha": "center", "va": "center"}

#     # Year/Month Label (+8)
#     ax.text(0.98, 0.98, f"{year} {MONTH_LABEL}", transform=ax.transAxes, ha="right", va="top",
#             fontsize=PANEL_FONT + 8, color="black", fontweight="normal", zorder=30)
            
#     # Panel Letter a-i (+8)
#     ax.text(0.02, 0.03, f"({panel_letter})", transform=ax.transAxes, ha="left", va="bottom",
#             fontsize=PANEL_FONT + 8, color="black", fontweight="bold", zorder=30)

#     # Bold borders
#     ax.spines["geo"].set_linewidth(1.2)
#     ax.spines["geo"].set_edgecolor("black")

#     print(f"INFO  [{month_str}] Panel {panel_letter} OK")

# # =========================================================
# # Figure-level annotation
# # =========================================================
# # 🎯 TITLE FIX: Removed reference text
# fig.suptitle(
#     f"Greater Bay Area \u2014 Sentinel-1 Monthly Composite ({MONTH_LABEL}, {YEARS_PLOT[-1]}\u2013{YEARS_PLOT[0]})\n"
#     f"R={BAND_NAMES[0]}  G={BAND_NAMES[1]}  B={BAND_NAMES[2]}  "
#     f"| Orbit: {DIRECTION} \n"
#     f"Stretch: p=[{P_MIN},{P_MAX}], gamma={GAMMA}\n",
#     fontsize=PANEL_FONT + 6,
#     fontweight="bold"
# )

# # =========================================================
# # Save
# # =========================================================
# save_dpi = 72 if DRAFT_MODE else FINAL_DPI
# suffix = "DRAFT" if DRAFT_MODE else "FINAL"
# out_png = Path(FIGURE_DIR) / f"{Study_Area}_{MONTH_LABEL}_{YEARS_PLOT[0]}-{YEARS_PLOT[-1]}_{DIRECTION}_4x3_Master_{suffix}.png"

# print(f"\n⏳ Rendering and saving to disk (DPI: {save_dpi}) ...")
# fig.savefig(
#     out_png,
#     dpi=save_dpi,
#     bbox_inches="tight",
#     facecolor="white",
#     pad_inches=0.03
# )

# t_end = time.time()
# print(f"✅ Done! Total elapsed time: {t_end - t_start:.2f} seconds")
# print(f"📂 Figure saved: {out_png}")

# plt.show()

In [10]:
# # =============================================================================
# # Asymmetrical 4x3 Grid Layout (All 9 Years)
# #
# # Layout Structure (Revised Order):
# # - ax_a (Top-Left)    : 2026
# # - ax_b (Mid-Left)    : 2025
# # - ax_c (Big-Right)   : 2024 (Spans rows 0-1, cols 1-2)
# # - ax_d to ax_i       : 2023 to 2018 (Rows 2-3)
# # =============================================================================
# import json as _json
# import numpy as np
# import matplotlib.pyplot as plt
# import matplotlib.ticker as mticker
# from matplotlib.gridspec import GridSpec
# import cartopy.crs as ccrs
# import cartopy.feature as cfeature
# from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
# from pathlib import Path
# from PIL import Image
# import geopandas as gpd
# import time

# # =========================================================
# # ⚙️ Core Layout Switch & Quality Settings
# # =========================================================
# DRAFT_MODE = False  # 设为 False 进行正式绘图
# FINAL_DPI = 600     

# PANEL_PNG_DIR = Path(FIGURE_DIR) / "panel_cache"
# plot_crs = ccrs.PlateCarree()

# print(f"\n{'='*50}")
# print(f"🚀 Assembling 4x3 Master Grid (Mode: {'[DRAFT]' if DRAFT_MODE else '[FINAL HIGH-RES]'})")
# print(f"{'='*50}")
# t_start = time.time()

# # 1. Load boundary
# boundary_gdf = None
# if BOUNDARY_SHP is not None:
#     boundary_gdf = gpd.read_file(BOUNDARY_SHP).to_crs("EPSG:4326")

# YEARS_PLOT = YEARS[::-1]  

# # 2. Initialize figure
# FIGSIZE4x3 = (15, 18) 
# fig = plt.figure(figsize=FIGSIZE4x3, facecolor="white", layout="compressed")
# gs = GridSpec(4, 3, figure=fig, wspace=0.0, hspace=0.0)

# ax_a = fig.add_subplot(gs[0, 0], projection=plot_crs)      
# ax_b = fig.add_subplot(gs[1, 0], projection=plot_crs)      
# ax_c = fig.add_subplot(gs[0:2, 1:3], projection=plot_crs)  

# ax_d = fig.add_subplot(gs[2, 0], projection=plot_crs)      
# ax_e = fig.add_subplot(gs[2, 1], projection=plot_crs)      
# ax_f = fig.add_subplot(gs[2, 2], projection=plot_crs)      
# ax_g = fig.add_subplot(gs[3, 0], projection=plot_crs)      
# ax_h = fig.add_subplot(gs[3, 1], projection=plot_crs)      
# ax_i = fig.add_subplot(gs[3, 2], projection=plot_crs)      

# axes_list = [ax_a, ax_b, ax_c, ax_d, ax_e, ax_f, ax_g, ax_h, ax_i]
# shared_extent_pad = None

# # 3. Render each panel
# for panel_idx, (ax, year) in enumerate(zip(axes_list, YEARS_PLOT)):
#     month_str = f"{year}-{MONTH}"
#     panel_letter = chr(97 + panel_idx)  

#     png_path  = PANEL_PNG_DIR / f"panel_{year}_{MONTH}_{DIRECTION}_{study_tag}.png"
#     json_path = PANEL_PNG_DIR / f"panel_{year}_{MONTH}_{DIRECTION}_{study_tag}.json"

#     if not png_path.exists() or not json_path.exists():
#         ax.text(0.5, 0.5, f"{year}-{MONTH}\nNo cache", ha="center", va="center", transform=ax.transAxes, fontsize=PANEL_FONT, color="red")
#         ax.axis("off")
#         continue

#     with open(json_path) as _f:
#         meta = _json.load(_f)
#     xmin, xmax, ymin, ymax = meta["extent"]
#     extent = [xmin, xmax, ymin, ymax]

#     if shared_extent_pad is None:
#         xpad = (xmax - xmin) * 0.03
#         ypad = (ymax - ymin) * 0.03
#         shared_extent_pad = [xmin - xpad, xmax + xpad, ymin - ypad, ymax + ypad]
    
#     if panel_idx == 2:
#         y_diff = shared_extent_pad[3] - shared_extent_pad[2]
#         expansion_factor = 0.012  
#         big_extent_pad = [shared_extent_pad[0], shared_extent_pad[1], shared_extent_pad[2] - (y_diff * expansion_factor), shared_extent_pad[3] + (y_diff * expansion_factor)]
#         ax.set_extent(big_extent_pad, crs=plot_crs)
#     else:
#         ax.set_extent(shared_extent_pad, crs=plot_crs)

#     if not DRAFT_MODE:
#         rgba = np.array(Image.open(str(png_path)).convert("RGBA"), dtype=np.float32) / 255.0
#         ax.add_feature(cfeature.OCEAN, facecolor="#cde8f5", zorder=1)
#         ax.add_feature(cfeature.LAND,  facecolor="#f5f5f0", zorder=2)
#         ax.imshow(rgba, extent=extent, transform=plot_crs, origin="upper", interpolation="bilinear", zorder=5)
#         ax.add_feature(cfeature.COASTLINE, linewidth=0.8, edgecolor="#333333", alpha=0.8, zorder=10)
#         if boundary_gdf is not None:
#             ax.add_geometries(boundary_gdf.geometry, crs=plot_crs, facecolor="none", edgecolor="black", linewidth=0.6, zorder=15)
#     else:
#         ax.set_facecolor("#eeeeee")
#         ax.text(0.5, 0.5, "DRAFT", transform=ax.transAxes, ha="center", va="center", fontsize=40, color="grey", alpha=0.4, zorder=5)

#     # Gridlines
#     gl = ax.gridlines(draw_labels=True, linewidth=0.5, color="grey", alpha=0.5, linestyle="--", zorder=20)
#     gl.ypadding, gl.xpadding = 15, 5
#     gl.top_labels = gl.bottom_labels = gl.left_labels = gl.right_labels = False

#     if panel_idx == 0: gl.top_labels = gl.left_labels = True
#     elif panel_idx == 1: gl.left_labels = True
#     elif panel_idx == 2: gl.top_labels = gl.right_labels = True
#     elif panel_idx == 3: gl.left_labels = True
#     elif panel_idx == 5: gl.right_labels = True
#     elif panel_idx == 6: gl.bottom_labels = gl.left_labels = True
#     elif panel_idx == 7: gl.bottom_labels = True
#     elif panel_idx == 8: gl.bottom_labels = gl.right_labels = True

#     gl.xlocator, gl.ylocator = mticker.MultipleLocator(1.0), mticker.MultipleLocator(1.0)
#     gl.xformatter, gl.yformatter = LONGITUDE_FORMATTER, LATITUDE_FORMATTER
#     gl.xlabel_style = {"size": (PANEL_FONT - 4) * 1.8}
#     gl.ylabel_style = {"size": (PANEL_FONT - 4) * 1.8, "rotation": 90, "ha": "center", "va": "center"}

#     # Date Label (Top-Right)
#     ax.text(0.98, 0.98, f"{year} {MONTH_LABEL}", transform=ax.transAxes, ha="right", va="top",
#             fontsize=PANEL_FONT + 8, color="black", fontweight="normal", zorder=30)
            
#     # ✅ Panel Letter (Bottom-Right) - 已修改
#     ax.text(0.98, 0.03, f"({panel_letter})", transform=ax.transAxes, ha="right", va="bottom",
#             fontsize=PANEL_FONT + 8, color="black", fontweight="bold", zorder=30)

#     ax.spines["geo"].set_linewidth(1.2)
#     ax.spines["geo"].set_edgecolor("black")
#     print(f"INFO  [{month_str}] Panel {panel_letter} OK")

# # 4. Title & Save
# fig.suptitle(f"Greater Bay Area \u2014 Sentinel-1 Monthly Composite ({MONTH_LABEL}, {YEARS_PLOT[-1]}\u2013{YEARS_PLOT[0]})\n"
#              f"R={BAND_NAMES[0]}  G={BAND_NAMES[1]}  B={BAND_NAMES[2]}  | Orbit: {DIRECTION} \n"
#              f"Stretch: p=[{P_MIN},{P_MAX}], gamma={GAMMA}\n", fontsize=PANEL_FONT + 6, fontweight="bold")

# suffix = "DRAFT" if DRAFT_MODE else "FINAL"
# out_png = Path(FIGURE_DIR) / f"{Study_Area}_{MONTH_LABEL}_{YEARS_PLOT[0]}-{YEARS_PLOT[-1]}_{DIRECTION}_4x3_Master_{suffix}.png"
# fig.savefig(out_png, dpi=save_dpi, bbox_inches="tight", facecolor="white", pad_inches=0.03)

# print(f"✅ Done! Figure saved: {out_png}")
# plt.show()

## Only 2*3

In [11]:
# =============================================================================
# Asymmetrical 3-Panel Layout (2017, 2021, 2026)
#
# Layout Structure:
# - ax_a (Top-Left, Small)    : 2017
# - ax_b (Bottom-Left, Small) : 2021
# - ax_c (Right, Big)         : 2026 (Spans both rows)
#
# Optimizations Included:
# - FINAL_DPI = 600 for publication quality.
# - interpolation="none" to preserve raw 30m resolution sharpness.
# - gl.ypadding=15, gl.xpadding=4 for perfect axis label spacing.
# - Panel letters (a, b, c) placed in the bottom-right corner.
# =============================================================================
import json as _json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from pathlib import Path
from PIL import Image
import geopandas as gpd
import time

# =========================================================
# ⚙️ Core Layout Switch & Quality Settings
# =========================================================
DRAFT_MODE = False  # 设为 False 进行正式 600 DPI 绘图
FINAL_DPI = 600     

PANEL_PNG_DIR = Path(FIGURE_DIR) / "panel_cache"
plot_crs = ccrs.PlateCarree()

print(f"\n{'='*50}")
print(f"🚀 Assembling 3-Panel Asymmetrical Grid (Mode: {'[DRAFT]' if DRAFT_MODE else '[FINAL HIGH-RES]'})")
print(f"{'='*50}")
t_start = time.time()

# 1. Load boundary
boundary_gdf = None
if BOUNDARY_SHP is not None:
    boundary_gdf = gpd.read_file(BOUNDARY_SHP).to_crs("EPSG:4326")

# ✅ 强制指定要绘制的 3 个特定年份
YEARS_PLOT = [2017, 2021, 2026]  

# 2. Initialize figure
# 调整画布比例适应 2小+1大 的结构
FIGSIZE_3PANEL = (14, 9) 
fig = plt.figure(figsize=FIGSIZE_3PANEL, facecolor="white", layout="compressed")

# 使用 width_ratios=[1.0, 2.0] 保证大图长宽比正确
gs = GridSpec(2, 2, figure=fig, width_ratios=[1.0, 2.0], wspace=0.0, hspace=0.0)

ax_a = fig.add_subplot(gs[0, 0], projection=plot_crs)      # 2017: Top-Left
ax_b = fig.add_subplot(gs[1, 0], projection=plot_crs)      # 2021: Bottom-Left
ax_c = fig.add_subplot(gs[:, 1], projection=plot_crs)      # 2026: Big-Right (spans both rows)

axes_list = [ax_a, ax_b, ax_c]
shared_extent_pad = None

# 3. Render each panel
for panel_idx, (ax, year) in enumerate(zip(axes_list, YEARS_PLOT)):
    month_str = f"{year}-{MONTH}"
    panel_letter = chr(97 + panel_idx)  

    png_path  = PANEL_PNG_DIR / f"panel_{year}_{MONTH}_{DIRECTION}_{study_tag}.png"
    json_path = PANEL_PNG_DIR / f"panel_{year}_{MONTH}_{DIRECTION}_{study_tag}.json"

    if not png_path.exists() or not json_path.exists():
        ax.text(0.5, 0.5, f"{year}-{MONTH}\nNo cache", ha="center", va="center", transform=ax.transAxes, fontsize=PANEL_FONT, color="red")
        ax.axis("off")
        continue

    with open(json_path) as _f:
        meta = _json.load(_f)
    xmin, xmax, ymin, ymax = meta["extent"]
    extent = [xmin, xmax, ymin, ymax]

    # 锁定全局边界
    if shared_extent_pad is None:
        xpad = (xmax - xmin) * 0.03
        ypad = (ymax - ymin) * 0.03
        shared_extent_pad = [xmin - xpad, xmax + xpad, ymin - ypad, ymax + ypad]
    
    # 🎯 ALIGNMENT FIX: 稍微拉伸大图的 Y 轴边界以补偿网格缝隙，使上下边缘完美对齐
    if panel_idx == 2:
        y_diff = shared_extent_pad[3] - shared_extent_pad[2]
        expansion_factor = 0.012  
        big_extent_pad = [shared_extent_pad[0], shared_extent_pad[1], shared_extent_pad[2] - (y_diff * expansion_factor), shared_extent_pad[3] + (y_diff * expansion_factor)]
        ax.set_extent(big_extent_pad, crs=plot_crs)
    else:
        ax.set_extent(shared_extent_pad, crs=plot_crs)

    # 渲染图像
    if not DRAFT_MODE:
        rgba = np.array(Image.open(str(png_path)).convert("RGBA"), dtype=np.float32) / 255.0
        ax.add_feature(cfeature.OCEAN, facecolor="#cde8f5", zorder=1)
        ax.add_feature(cfeature.LAND,  facecolor="#f5f5f0", zorder=2)
        # 强制使用 interpolation="none" 保证高分辨率下像素依然锐利
        ax.imshow(rgba, extent=extent, transform=plot_crs, origin="upper", interpolation="none", zorder=5)
        ax.add_feature(cfeature.COASTLINE, linewidth=0.8, edgecolor="#333333", alpha=0.8, zorder=10)
        if boundary_gdf is not None:
            ax.add_geometries(boundary_gdf.geometry, crs=plot_crs, facecolor="none", edgecolor="black", linewidth=0.6, zorder=15)
    else:
        ax.set_facecolor("#eeeeee")
        ax.text(0.5, 0.5, "DRAFT", transform=ax.transAxes, ha="center", va="center", fontsize=40, color="grey", alpha=0.4, zorder=5)

    # Gridlines
    gl = ax.gridlines(draw_labels=True, linewidth=0.5, color="grey", alpha=0.5, linestyle="--", zorder=20)
    
    # 调整坐标轴与边框的距离
    gl.ypadding = 15  
    gl.xpadding = 4   
    
    gl.top_labels = gl.bottom_labels = gl.left_labels = gl.right_labels = False

    # 针对 a, b, c 的位置智能分配坐标轴标签
    if panel_idx == 0:     # a (2017, Top-Left)
        gl.top_labels = True
        gl.left_labels = True
    elif panel_idx == 1:   # b (2021, Bottom-Left)
        gl.bottom_labels = True
        gl.left_labels = True
    elif panel_idx == 2:   # c (2026, Big-Right)
        gl.top_labels = True
        gl.bottom_labels = True
        gl.right_labels = True

    gl.xlocator, gl.ylocator = mticker.MultipleLocator(1.0), mticker.MultipleLocator(1.0)
    gl.xformatter, gl.yformatter = LONGITUDE_FORMATTER, LATITUDE_FORMATTER
    
    # 缩小坐标轴字体乘数以适应新的画幅比例
    gl.xlabel_style = {"size": (PANEL_FONT - 4) * 1.4}
    gl.ylabel_style = {"size": (PANEL_FONT - 4) * 1.4, "rotation": 90, "ha": "center", "va": "center"}

    # Date Label (Top-Right)
    ax.text(0.98, 0.98, f"{year} {MONTH_LABEL}", transform=ax.transAxes, ha="right", va="top",
            fontsize=PANEL_FONT + 6, color="black", fontweight="normal", zorder=30)
            
    # ✅ Panel Letter (Bottom-Right)
    ax.text(0.98, 0.03, f"({panel_letter})", transform=ax.transAxes, ha="right", va="bottom",
            fontsize=PANEL_FONT + 6, color="black", fontweight="bold", zorder=30)

    ax.spines["geo"].set_linewidth(1.2)
    ax.spines["geo"].set_edgecolor("black")
    print(f"INFO  [{month_str}] Panel {panel_letter} OK")

# 4. Title & Save
fig.suptitle(f"Greater Bay Area \u2014 Sentinel-1 Monthly Composite ({MONTH_LABEL}, 2017, 2021, 2026)\n"
             f"R={BAND_NAMES[0]}  G={BAND_NAMES[1]}  B={BAND_NAMES[2]}  | Orbit: {DIRECTION} \n"
             f"Stretch: p=[{P_MIN},{P_MAX}], gamma={GAMMA}\n", fontsize=PANEL_FONT + 6, fontweight="bold")

suffix = "DRAFT" if DRAFT_MODE else "FINAL"
out_png = Path(FIGURE_DIR) / f"{Study_Area}_{MONTH_LABEL}_2017_2021_2026_{DIRECTION}_3_panels_{suffix}.png"

print(f"\n⏳ Rendering and saving to disk (DPI: {FINAL_DPI}) ... This may take a few minutes...")
fig.savefig(out_png, dpi=FINAL_DPI, bbox_inches="tight", facecolor="white", pad_inches=0.03)

print(f"✅ Done! Figure saved: {out_png}")
plt.show()


🚀 Assembling 3-Panel Asymmetrical Grid (Mode: [FINAL HIGH-RES])
INFO  [2017-03] Panel a OK
INFO  [2021-03] Panel b OK
INFO  [2026-03] Panel c OK

⏳ Rendering and saving to disk (DPI: 600) ... This may take a few minutes...
✅ Done! Figure saved: D:\QGIS\S1-GRiTS-Figures\Figure_PortHedland_AUS\GBA_Mar_2017_2021_2026_DESCENDING_3_panels_FINAL.png


<Figure size 1400x900 with 3 Axes>